# 06 · Optimise **every crease and every angle** with `EvoTF_ES` on Warp **[GPU]**

Google Colab notebook: needs an NVIDIA T4 GPU (`Runtime → Change runtime type → T4 GPU`).

It builds a 2×2 Miura design space with **4 hinge angles + 8 crease-vertex coordinates = 12 free variables**, scores every candidate with MuJoCo **Warp** on the GPU (one world per genome in a single batched model), and drives the search with `evosax`'s `EvoTF_ES`. The best genome of every generation is saved as crease-pattern JSON.

The score is **85 % in-plane movement + 15 % fold-to-target**. See `docs/COLAB_GPU.md` for measured timings.

## 1. Install **[GPU]**

Select a T4 GPU runtime before running this cell. The pin installs the released version with the Warp and optimiser extras.

In [ ]:
%pip install -q --no-cache-dir "origami-engine-mjx[warp,opt]==1.3.0"

In [ ]:
# Verify GPU + backend (a helpful error instead of a crash on CPU runtimes)
import subprocess
from pathlib import Path

import jax
import jax.extend.core as _jxcore

# evosax 0.3.2 uses a JAX API removed in 0.11; expose it for compatibility.
if not hasattr(jax.core, "get_opaque_trace_state"):
    jax.core.get_opaque_trace_state = _jxcore.get_opaque_trace_state
import jax.numpy as jnp
import numpy as np
from evosax.algorithms import algorithms as EVO_ALGORITHMS

import origami_engine_mjx as oem
from origami_engine_mjx import Crease, save_creases_to_json
from origami_engine_mjx.backends import available_backends
from origami_engine_mjx.integrations import (
    DesignSpace,
    GenerationLogger,
    OrigamiEvaluator,
    load_seeds,
    mixed_population,
)

if subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, check=False).returncode != 0:
    raise RuntimeError("No NVIDIA GPU found. In Colab: Runtime -> Change runtime type -> T4 GPU, then rerun all.")
if "warp" not in available_backends():
    raise RuntimeError("Warp backend unavailable. Re-run the install cell on a GPU runtime.")
print("origami-engine-mjx", oem.__version__, "| backends", available_backends())
print("GPU:", subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, check=False).stdout.strip())
print("jax", jax.__version__, "devices", jax.devices())

In [ ]:
# ---- settings ---------------------------------------------------------
STRATEGY = "EvoTF_ES"
MOVEMENT_WEIGHT, FOLD_WEIGHT = 85.0, 15.0
POP_SIZE, GENERATIONS, SEED = 32, 3, 0  # small budget; raise for real searches
ALL_CREASES = "all"  # angles AND every crease vertex
INIT_MODE, SEED_JSON = "mixed", "seeds"
EVAL_DURATION, EVAL_FOLD = 0.05, 0.02  # simulated seconds per candidate
OUT_DIR = "runs/evotf"
print(f"strategy={STRATEGY} pop={POP_SIZE} generations={GENERATIONS} optimising={ALL_CREASES}")

## 2. Design space: every crease, every angle

`DesignSpace.everything` exposes one variable per actuated hinge angle and one `x`/`y` coordinate per unique crease vertex. The two loop-closing creases follow the shared vertices.

In [ ]:
PATTERN = [
    Crease((0.1, 0.1), (0.1, 0.5), 0.8),
    Crease((0.1, 0.5), (0.5, 0.5), -0.8),
    Crease((0.5, 0.5), (0.5, 0.1), 0.8),
    Crease((0.5, 0.1), (0.1, 0.1), -0.8),
    Crease((0.1, 0.1), (0.5, 0.5), -0.8),  # loop closer
    Crease((0.1, 0.5), (0.5, 0.1), 0.8),  # loop closer
]
space = (
    DesignSpace.everything(PATTERN, name="miura-2x2", coord_limits=(0.0, 1.0))
    if ALL_CREASES == "all"
    else DesignSpace.angles_only(PATTERN, name="miura-2x2")
)
print(f"free variables (genome dim): {space.dim}")
print(f"  angles: {space.groups.count('angles')}   geometry: {space.groups.count('geometry')}")
for name, group in zip(space.parameter_names, space.groups):
    print(f"  {group:9s} {name}")

## 3. Seed points and initial population

In [ ]:
seed_dir = Path(SEED_JSON)
seed_dir.mkdir(parents=True, exist_ok=True)
flipped = [Crease(c.p1, c.p2, -c.angle) for c in PATTERN]
for name, pattern in (("seed_template.json", PATTERN), ("seed_flipped.json", flipped)):
    path = seed_dir / name
    if not path.exists():
        save_creases_to_json(pattern, path)
seeds = load_seeds(seed_dir, space)
rng = np.random.default_rng(SEED)
pop = mixed_population(
    space,
    POP_SIZE,
    seeds=seeds.genomes,
    fractions={"seed": 0.2, "interp": 0.5, "random": 0.3},
    rng=rng,
    valid_only=True,
)
Z0 = np.vstack([space.to_normalized(g) for g in pop.genomes])
print(f"initial population: {Z0.shape[0]} candidates x {space.dim} variables")

## 4. Objective: 85 % movement + 15 % fold

`OrigamiEvaluator` scores a whole population in one batched Warp model (one world per genome) inside a single CUDA graph. Higher fitness is better; `evosax` minimises, so we hand it `-fitness`.

In [ ]:
evaluator = OrigamiEvaluator(
    space,
    backend="warp",
    duration=EVAL_DURATION,
    fold_duration=EVAL_FOLD,
    vibrate=False,
    maximize=True,
    error_policy="penalty",
    movement_weight=MOVEMENT_WEIGHT,
    fold_weight=FOLD_WEIGHT,
)
record = evaluator.evaluate(space.base_vector())
print(f"template: fitness={record.fitness:+.4f} status={record.status}")
for k in ("displacement_x", "fold_score", "fold_mean_abs_error_deg"):
    print(f"  {k:26s} {record.descriptors.get(k)}")
advance = evaluator.to_batch_objective("minimize")

## 5. Build `EvoTF_ES`

In [ ]:
cls = EVO_ALGORITHMS[STRATEGY]
es = cls(population_size=POP_SIZE, solution=jnp.zeros((space.dim,), dtype=jnp.float32))
params = es.default_params
if STRATEGY == "EvoTF_ES":
    params = params.replace(std_init=1.0)
key = jax.random.key(SEED)
key, key_init = jax.random.split(key)
state = es.init(key_init, jnp.asarray(Z0.mean(axis=0), dtype=jnp.float32), params)
print("EvoTF_ES ready; dim =", space.dim, "pop =", POP_SIZE)

## 6. Run the search (best genome of every generation saved)

In [ ]:
import time

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
logger = GenerationLogger(
    out_dir / "evotf.jsonl",
    space,
    maximising=True,
    best_path=out_dir / "evotf.best.json",
    summary_path=out_dir / "evotf.summary.json",
    keep_every_best=True,
)
history, t_total = [], time.perf_counter()
for gen in range(1, GENERATIONS + 1):
    key, key_ask, key_tell = jax.random.split(key, 3)
    Z, state = es.ask(key_ask, state, params)
    Z = np.clip(np.asarray(Z, dtype=float), -1.0, 1.0)
    genomes = np.vstack([space.from_normalized(z) for z in Z])
    t0 = time.perf_counter()
    batch = evaluator.evaluate_batch(genomes)
    dt = time.perf_counter() - t0
    fitness = np.asarray(batch.fitness, dtype=float)
    records = batch.as_records()
    logger.log_batch(batch, genomes, generation=gen)
    summary = logger.end_generation(gen, records, genomes)
    history.append(summary.best_fitness)
    losses = -fitness
    losses[~np.isfinite(losses)] = np.inf
    state, _info = es.tell(key_tell, jnp.asarray(Z, dtype=jnp.float32), jnp.asarray(losses), state, params)
    print(
        f"gen {gen:3d}: best={summary.best_fitness:+.5f}  mean={summary.mean_fitness:+.5f}  ok={sum(r.status == 'ok' for r in records)}/{len(records)}  eval={dt:.3f}s ({len(genomes) / dt:.0f} cand/s)"
    )
print(f"\nglobal best: {logger.best_fitness:+.4f} at generation {logger.best_generation}")
print(f"{POP_SIZE * GENERATIONS} candidates in {time.perf_counter() - t_total:.1f}s total")

## 7. Inspect the winner

In [ ]:
best_genome = logger.best_genome
best_record = evaluator.evaluate(best_genome)
print(f"best fitness       : {best_record.fitness:+.4f}")
print(f"  movement (m)     : {best_record.descriptors.get('displacement_x')}")
print(f"  fold score (1=good): {best_record.descriptors.get('fold_score')}")
print(f"  mean hinge error : {best_record.descriptors.get('fold_mean_abs_error_deg')}")
print("best fitness per gen:", [round(x, 4) for x in history])
print("optimised design:")
for c in space.decode(best_genome):
    print(f"  {np.round(c.p1, 3)} -> {np.round(c.p2, 3)}   angle {np.rad2deg(c.angle):+7.2f} deg")

In [ ]:
print("files written:")
for path in sorted(out_dir.glob("evotf*")):
    print(f"  {path.name:32s} {path.stat().st_size:>8,} bytes")
print(f"{len(list(out_dir.glob('evotf.best_gen_*.json')))} per-generation checkpoints")